# V4 Model Inference - Optimized for Maximum LB Score

**Current LB: 0.550 → Target: 0.58+**

## Optimization Strategies
1. **Overlap 0.5** - Reduces tile artifacts → better VOI
2. **Threshold sweep** - Find optimal threshold (not 0.5)
3. **Hysteresis thresholding** - Preserves thin connections → better TopoScore
4. **Small component removal** - Removes noise → better SurfaceDice
5. **TTA (Test Time Augmentation)** - Optional flip averaging

---

In [ ]:
!pip install imagecodecs connected-components-3d tifffile -q

In [ ]:
# =============================================================================
# CELL 1: IMPORTS & CONFIG
# =============================================================================

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import gc
import json
import zipfile
from pathlib import Path
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd
import tifffile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast

from scipy import ndimage
from scipy.ndimage import binary_dilation, distance_transform_edt

try:
    import cc3d
    USE_CC3D = True
except:
    USE_CC3D = False

# =============================================================================
# INFERENCE CONFIG
# =============================================================================

class InferenceConfig:
    # Paths - UPDATE THESE
    CHECKPOINT_PATH = Path("/kaggle/input/v4-checkpoint/best_model.pth")
    TEST_IMAGES_DIR = Path("/kaggle/input/vesuvius-challenge-2025/test")
    OUTPUT_DIR = Path("/kaggle/working/predictions")
    
    # Model config (must match training)
    PATCH_SIZE = (128, 128, 128)
    FEATURES = [32, 64, 128, 256, 320, 320]
    N_BLOCKS = [1, 3, 4, 6, 6, 6]
    
    # Inference settings
    OVERLAP = 0.5              # 50% overlap for better stitching
    USE_AMP = True
    USE_TTA = False            # Test-time augmentation (slower but can help)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    
    # ==========================================================================
    # POSTPROCESSING - KEY FOR LB IMPROVEMENT
    # ==========================================================================
    THRESHOLD = 0.30           # Lower than 0.5 to preserve thin structures
    USE_HYSTERESIS = True
    HYSTERESIS_LOW = 0.16
    HYSTERESIS_HIGH = 0.38
    MIN_COMPONENT_SIZE = 50    # Remove small noise
    CLOSING_ITERS = 0          # 0 or 1 (careful: can bridge gaps)

cfg = InferenceConfig()

print("="*70)
print("V4 INFERENCE - OPTIMIZED FOR LB")
print("="*70)
print(f"Patch: {cfg.PATCH_SIZE} | Overlap: {cfg.OVERLAP}")
print(f"Threshold: {cfg.THRESHOLD}")
print(f"Hysteresis: {cfg.USE_HYSTERESIS} ({cfg.HYSTERESIS_LOW}, {cfg.HYSTERESIS_HIGH})")
print(f"Min component: {cfg.MIN_COMPONENT_SIZE}")
print(f"TTA: {cfg.USE_TTA}")
print("="*70)

In [ ]:
# =============================================================================
# CELL 2: MODEL DEFINITION (V4 architecture)
# =============================================================================

class SafeInstanceNorm3d(nn.Module):
    def __init__(self, num_features, eps=1e-5, affine=True):
        super().__init__()
        self.num_features = num_features
        self.eps = eps
        self.affine = affine
        if affine:
            self.weight = nn.Parameter(torch.ones(num_features))
            self.bias = nn.Parameter(torch.zeros(num_features))
    
    def forward(self, x):
        mean = x.mean(dim=[2,3,4], keepdim=True)
        var = x.var(dim=[2,3,4], keepdim=True, unbiased=False)
        var_safe = torch.clamp(var, min=self.eps)
        x_norm = (x - mean) / torch.sqrt(var_safe + self.eps)
        if self.affine:
            x_norm = self.weight.view(1,-1,1,1,1) * x_norm + self.bias.view(1,-1,1,1,1)
        return x_norm


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=True),
            SafeInstanceNorm3d(out_ch),
            nn.LeakyReLU(0.01, inplace=True),
        )
    
    def forward(self, x):
        return self.conv(x)


class ResBlock(nn.Module):
    def __init__(self, channels, n_convs=2):
        super().__init__()
        self.blocks = nn.Sequential(*[ConvBlock(channels, channels) for _ in range(n_convs)])
    
    def forward(self, x):
        return x + self.blocks(x)


class scSEBlock(nn.Module):
    def __init__(self, channels, reduction=2):
        super().__init__()
        self.cse_pool = nn.AdaptiveAvgPool3d(1)
        self.cse_fc1 = nn.Linear(channels, channels // reduction)
        self.cse_fc2 = nn.Linear(channels // reduction, channels)
        self.sse_conv = nn.Conv3d(channels, 1, 1)
    
    def forward(self, x):
        b, c = x.shape[:2]
        cse = torch.sigmoid(self.cse_fc2(F.relu(self.cse_fc1(self.cse_pool(x).view(b,c))))).view(b,c,1,1,1)
        sse = torch.sigmoid(self.sse_conv(x))
        return x * cse + x * sse


class ResEncUNet3D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, features=None, n_blocks=None,
                 use_scse=True, use_deep_supervision=False):
        super().__init__()
        
        if features is None:
            features = [32, 64, 128, 256, 320, 320]
        if n_blocks is None:
            n_blocks = [1, 3, 4, 6, 6, 6]
        
        self.features = features
        self.use_scse = use_scse
        self.use_deep_supervision = use_deep_supervision
        
        # Encoder
        self.encoders = nn.ModuleList()
        self.attentions = nn.ModuleList()
        self.pools = nn.ModuleList()
        
        for i, (feat, nb) in enumerate(zip(features, n_blocks)):
            in_channels = in_ch if i == 0 else features[i-1]
            self.encoders.append(nn.Sequential(
                ConvBlock(in_channels, feat),
                *[ResBlock(feat) for _ in range(nb)]
            ))
            self.attentions.append(scSEBlock(feat) if use_scse else nn.Identity())
            if i < len(features) - 1:
                self.pools.append(nn.Conv3d(feat, feat, 2, stride=2, bias=True))
        
        # Decoder
        self.ups = nn.ModuleList()
        self.dec_convs = nn.ModuleList()
        
        for i in range(len(features)-2, -1, -1):
            self.ups.append(nn.ConvTranspose3d(features[i+1], features[i], 2, stride=2, bias=True))
            self.dec_convs.append(ConvBlock(features[i]*2, features[i]))
        
        self.final = nn.Conv3d(features[0], out_ch, 1, bias=True)
    
    def forward(self, x):
        enc_features = []
        for i, (enc, att) in enumerate(zip(self.encoders, self.attentions)):
            x = enc(x)
            x = att(x)
            enc_features.append(x)
            if i < len(self.pools):
                x = self.pools[i](x)
        
        enc_features = enc_features[::-1]
        x = enc_features[0]
        
        for i, (up, dec) in enumerate(zip(self.ups, self.dec_convs)):
            x = up(x)
            skip = enc_features[i+1]
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = torch.cat([x, skip], dim=1)
            x = dec(x)
        
        return self.final(x)

print("Model defined")

In [ ]:
# =============================================================================
# CELL 3: LOAD MODEL
# =============================================================================

def load_model(checkpoint_path: Path, device: str = 'cuda'):
    """Load V4 model from checkpoint."""
    print(f"Loading model from {checkpoint_path}")
    
    model = ResEncUNet3D(
        features=cfg.FEATURES,
        n_blocks=cfg.N_BLOCKS,
        use_scse=True,
        use_deep_supervision=False,  # Disable for inference
    ).to(device)
    
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    
    # Handle compiled model state dict
    state_dict = ckpt['model_state_dict']
    state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}
    
    model.load_state_dict(state_dict, strict=False)
    model.eval()
    
    print(f"Loaded from epoch {ckpt.get('epoch', 'unknown')}")
    print(f"Best score: {ckpt.get('best_score', 'unknown')}")
    
    return model

# Load model
if cfg.CHECKPOINT_PATH.exists():
    model = load_model(cfg.CHECKPOINT_PATH, cfg.DEVICE)
else:
    print(f"Checkpoint not found: {cfg.CHECKPOINT_PATH}")
    print("Please update cfg.CHECKPOINT_PATH")

In [ ]:
# =============================================================================
# CELL 4: INFERENCE FUNCTIONS
# =============================================================================

@torch.no_grad()
def predict_volume(
    model,
    volume: np.ndarray,
    patch_size: Tuple[int,int,int] = (128,128,128),
    overlap: float = 0.5,
    device: str = 'cuda',
    use_tta: bool = False,
) -> np.ndarray:
    """
    Predict full volume with overlapping patches and Gaussian weighting.
    
    Args:
        model: Trained model
        volume: Input volume (D, H, W)
        patch_size: Patch size
        overlap: Overlap fraction (0.5 = 50%)
        device: Device
        use_tta: Use test-time augmentation (z-flip)
    
    Returns:
        Probability map (D, H, W)
    """
    model.eval()
    D, H, W = volume.shape
    pd, ph, pw = patch_size
    
    # Stride
    sd = int(pd * (1 - overlap))
    sh = int(ph * (1 - overlap))
    sw = int(pw * (1 - overlap))
    
    # Output arrays
    pred_sum = np.zeros((D, H, W), dtype=np.float32)
    count = np.zeros((D, H, W), dtype=np.float32)
    
    # Gaussian weighting for smooth blending
    sigma = 0.125
    gz = np.exp(-0.5 * ((np.arange(pd) - pd/2) / (pd * sigma))**2)
    gy = np.exp(-0.5 * ((np.arange(ph) - ph/2) / (ph * sigma))**2)
    gx = np.exp(-0.5 * ((np.arange(pw) - pw/2) / (pw * sigma))**2)
    gauss = (gz[:, None, None] * gy[None, :, None] * gx[None, None, :]).astype(np.float32)
    
    # Compute patch positions
    z_positions = list(range(0, max(1, D - pd + 1), sd))
    if D > pd and (D - pd) not in z_positions:
        z_positions.append(D - pd)
    
    y_positions = list(range(0, max(1, H - ph + 1), sh))
    if H > ph and (H - ph) not in y_positions:
        y_positions.append(H - ph)
    
    x_positions = list(range(0, max(1, W - pw + 1), sw))
    if W > pw and (W - pw) not in x_positions:
        x_positions.append(W - pw)
    
    # Normalize volume
    vol_norm = (volume.astype(np.float32) - volume.mean()) / (volume.std() + 1e-8)
    
    total_patches = len(z_positions) * len(y_positions) * len(x_positions)
    print(f"Processing {total_patches} patches (overlap={overlap})")
    
    pbar = tqdm(total=total_patches, desc="Inference")
    
    for z in z_positions:
        for y in y_positions:
            for x in x_positions:
                # Extract patch
                patch = vol_norm[z:z+pd, y:y+ph, x:x+pw].copy()
                actual_shape = patch.shape
                
                # Pad if needed
                if patch.shape != (pd, ph, pw):
                    pad_d = pd - patch.shape[0]
                    pad_h = ph - patch.shape[1]
                    pad_w = pw - patch.shape[2]
                    patch = np.pad(patch, [(0, pad_d), (0, pad_h), (0, pad_w)], mode='reflect')
                
                # To tensor
                inp = torch.from_numpy(patch[None, None]).float().to(device)
                
                # Predict
                with autocast(enabled=cfg.USE_AMP):
                    out = model(inp)
                    pred = torch.sigmoid(out).squeeze().cpu().numpy()
                
                # TTA: z-flip
                if use_tta:
                    inp_flip = torch.flip(inp, dims=[2])  # Flip z
                    with autocast(enabled=cfg.USE_AMP):
                        out_flip = model(inp_flip)
                        pred_flip = torch.sigmoid(out_flip).squeeze().cpu().numpy()
                    pred_flip = np.flip(pred_flip, axis=0)
                    pred = (pred + pred_flip) / 2
                
                # Crop to actual size
                pred_crop = pred[:actual_shape[0], :actual_shape[1], :actual_shape[2]]
                gauss_crop = gauss[:actual_shape[0], :actual_shape[1], :actual_shape[2]]
                
                # Accumulate
                pred_sum[z:z+actual_shape[0], y:y+actual_shape[1], x:x+actual_shape[2]] += pred_crop * gauss_crop
                count[z:z+actual_shape[0], y:y+actual_shape[1], x:x+actual_shape[2]] += gauss_crop
                
                pbar.update(1)
    
    pbar.close()
    
    # Average
    prob_map = pred_sum / np.maximum(count, 1e-8)
    
    return prob_map

print("Inference function ready")

In [ ]:
# =============================================================================
# CELL 5: POSTPROCESSING FUNCTIONS
# =============================================================================

def connected_components_3d(vol, connectivity=26):
    """3D connected components."""
    if USE_CC3D:
        return cc3d.connected_components(vol.astype(np.uint8), connectivity=connectivity)
    else:
        struct = ndimage.generate_binary_structure(3, 3 if connectivity == 26 else 1)
        labeled, _ = ndimage.label(vol, structure=struct)
        return labeled


def hysteresis_threshold(prob: np.ndarray, low: float, high: float, iterations: int = 5) -> np.ndarray:
    """
    Hysteresis thresholding.
    
    - Start with high-confidence voxels (prob >= high)
    - Grow to include connected low-confidence voxels (prob >= low)
    
    This preserves thin structures that connect high-confidence regions.
    """
    # Seed: high-confidence voxels
    binary = (prob >= high).astype(np.uint8)
    
    # Mask: all voxels above low threshold
    mask = (prob >= low)
    
    # Grow seed within mask
    for _ in range(iterations):
        binary = binary_dilation(binary) & mask
    
    return binary.astype(np.uint8)


def remove_small_components(binary: np.ndarray, min_size: int) -> np.ndarray:
    """Remove connected components smaller than min_size."""
    if min_size <= 0:
        return binary
    
    labeled = connected_components_3d(binary)
    result = np.zeros_like(binary)
    
    for i in range(1, labeled.max() + 1):
        component = (labeled == i)
        if component.sum() >= min_size:
            result[component] = 1
    
    return result


def postprocess_prediction(
    prob: np.ndarray,
    threshold: float = 0.30,
    use_hysteresis: bool = True,
    hysteresis_low: float = 0.16,
    hysteresis_high: float = 0.38,
    min_component_size: int = 50,
    closing_iters: int = 0,
) -> np.ndarray:
    """
    Full postprocessing pipeline optimized for LB.
    
    Args:
        prob: Probability map from model
        threshold: Simple threshold (used if hysteresis=False)
        use_hysteresis: Use hysteresis thresholding
        hysteresis_low: Low threshold for hysteresis
        hysteresis_high: High threshold for hysteresis
        min_component_size: Remove components smaller than this
        closing_iters: Morphological closing iterations (0 or 1)
    
    Returns:
        Binary segmentation (uint8)
    """
    print(f"Postprocessing: hyst={use_hysteresis}, min_size={min_component_size}, closing={closing_iters}")
    
    # Step 1: Thresholding
    if use_hysteresis:
        binary = hysteresis_threshold(prob, hysteresis_low, hysteresis_high)
        print(f"  Hysteresis ({hysteresis_low}, {hysteresis_high}): {100*binary.mean():.2f}% FG")
    else:
        binary = (prob >= threshold).astype(np.uint8)
        print(f"  Threshold {threshold}: {100*binary.mean():.2f}% FG")
    
    # Step 2: Remove small components
    if min_component_size > 0:
        n_before = connected_components_3d(binary).max()
        binary = remove_small_components(binary, min_component_size)
        n_after = connected_components_3d(binary).max()
        print(f"  Remove small: {n_before} -> {n_after} components")
    
    # Step 3: Morphological closing (optional)
    if closing_iters > 0:
        struct = ndimage.generate_binary_structure(3, 1)  # 6-connectivity (gentle)
        binary = ndimage.binary_closing(binary, structure=struct, iterations=closing_iters)
        binary = binary.astype(np.uint8)
        print(f"  After closing: {100*binary.mean():.2f}% FG")
    
    print(f"  Final: {100*binary.mean():.2f}% FG, {connected_components_3d(binary).max()} components")
    
    return binary

print("Postprocessing functions ready")

In [ ]:
# =============================================================================
# CELL 6: THRESHOLD SWEEP (for validation data)
# =============================================================================

def sweep_thresholds(
    prob: np.ndarray,
    gt: np.ndarray,
    thresholds: List[float] = None,
) -> Dict[float, float]:
    """
    Sweep thresholds to find optimal Dice.
    
    Use this on validation data to find best threshold.
    """
    if thresholds is None:
        thresholds = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
    
    gt_bin = (gt == 1).astype(np.uint8)
    ignore = (gt == 2)
    
    results = {}
    
    for thresh in thresholds:
        pred = (prob >= thresh).astype(np.uint8)
        pred[ignore] = 0
        gt_masked = gt_bin.copy()
        gt_masked[ignore] = 0
        
        inter = (pred & gt_masked).sum()
        union = pred.sum() + gt_masked.sum()
        dice = (2 * inter + 1e-5) / (union + 1e-5)
        
        results[thresh] = dice
        print(f"  Threshold {thresh:.2f}: Dice = {dice:.4f}")
    
    best_thresh = max(results, key=results.get)
    print(f"\nBest threshold: {best_thresh} (Dice = {results[best_thresh]:.4f})")
    
    return results


def sweep_hysteresis(
    prob: np.ndarray,
    gt: np.ndarray,
) -> Dict:
    """
    Sweep hysteresis parameters.
    """
    gt_bin = (gt == 1).astype(np.uint8)
    ignore = (gt == 2)
    
    configs = [
        (0.15, 0.35),
        (0.16, 0.38),
        (0.18, 0.40),
        (0.20, 0.40),
        (0.15, 0.45),
    ]
    
    results = {}
    
    for low, high in configs:
        pred = hysteresis_threshold(prob, low, high)
        pred[ignore] = 0
        gt_masked = gt_bin.copy()
        gt_masked[ignore] = 0
        
        inter = (pred & gt_masked).sum()
        union = pred.sum() + gt_masked.sum()
        dice = (2 * inter + 1e-5) / (union + 1e-5)
        
        results[(low, high)] = dice
        print(f"  Hysteresis ({low}, {high}): Dice = {dice:.4f}")
    
    best_config = max(results, key=results.get)
    print(f"\nBest hysteresis: {best_config} (Dice = {results[best_config]:.4f})")
    
    return results

print("Threshold sweep functions ready")

In [ ]:
# =============================================================================
# CELL 7: RUN ON VALIDATION (to find best params)
# =============================================================================

def run_validation_sweep(model, val_images_dir, val_labels_dir, num_volumes=3):
    """
    Run inference on validation volumes and sweep parameters.
    """
    val_images_dir = Path(val_images_dir)
    val_labels_dir = Path(val_labels_dir)
    
    vol_files = sorted(val_images_dir.glob("*.tif"))[:num_volumes]
    
    all_probs = []
    all_gts = []
    
    for vol_file in vol_files:
        vid = vol_file.stem
        print(f"\nProcessing {vid}...")
        
        # Load
        volume = tifffile.imread(vol_file)
        label = tifffile.imread(val_labels_dir / f"{vid}.tif")
        
        # Predict
        prob = predict_volume(
            model, volume,
            patch_size=cfg.PATCH_SIZE,
            overlap=cfg.OVERLAP,
            device=cfg.DEVICE,
            use_tta=cfg.USE_TTA,
        )
        
        all_probs.append(prob)
        all_gts.append(label)
    
    # Concatenate for global threshold sweep
    print("\n" + "="*50)
    print("THRESHOLD SWEEP")
    print("="*50)
    
    for i, (prob, gt) in enumerate(zip(all_probs, all_gts)):
        print(f"\nVolume {i}:")
        sweep_thresholds(prob, gt)
    
    print("\n" + "="*50)
    print("HYSTERESIS SWEEP")
    print("="*50)
    
    for i, (prob, gt) in enumerate(zip(all_probs, all_gts)):
        print(f"\nVolume {i}:")
        sweep_hysteresis(prob, gt)
    
    return all_probs, all_gts

# Uncomment to run validation sweep:
# probs, gts = run_validation_sweep(
#     model,
#     "/kaggle/input/3d-volume-training-data/train_images",
#     "/kaggle/input/3d-volume-training-data/train_labels",
#     num_volumes=3
# )

print("Validation sweep ready")

In [ ]:
# =============================================================================
# CELL 8: GENERATE SUBMISSION
# =============================================================================

def generate_submission(
    model,
    test_images_dir: Path,
    output_dir: Path,
    postprocess_params: Dict = None,
) -> Path:
    """
    Generate submission.zip
    
    Args:
        model: Trained model
        test_images_dir: Directory with test volumes
        output_dir: Output directory
        postprocess_params: Postprocessing parameters (or use defaults)
    
    Returns:
        Path to submission.zip
    """
    test_images_dir = Path(test_images_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Default postprocess params
    if postprocess_params is None:
        postprocess_params = {
            'threshold': cfg.THRESHOLD,
            'use_hysteresis': cfg.USE_HYSTERESIS,
            'hysteresis_low': cfg.HYSTERESIS_LOW,
            'hysteresis_high': cfg.HYSTERESIS_HIGH,
            'min_component_size': cfg.MIN_COMPONENT_SIZE,
            'closing_iters': cfg.CLOSING_ITERS,
        }
    
    print("Postprocess params:")
    for k, v in postprocess_params.items():
        print(f"  {k}: {v}")
    
    # Get test volumes
    test_files = sorted(test_images_dir.glob("*.tif"))
    print(f"\nFound {len(test_files)} test volumes")
    
    # Process each volume
    for vol_file in test_files:
        vid = vol_file.stem
        print(f"\n{'='*50}")
        print(f"Processing {vid}")
        print(f"{'='*50}")
        
        # Load
        volume = tifffile.imread(vol_file)
        print(f"Volume shape: {volume.shape}")
        
        # Predict
        prob = predict_volume(
            model, volume,
            patch_size=cfg.PATCH_SIZE,
            overlap=cfg.OVERLAP,
            device=cfg.DEVICE,
            use_tta=cfg.USE_TTA,
        )
        
        # Postprocess
        binary = postprocess_prediction(
            prob,
            **postprocess_params,
        )
        
        # Save
        out_path = output_dir / f"{vid}.tif"
        tifffile.imwrite(out_path, binary.astype(np.uint8))
        print(f"Saved to {out_path}")
        
        # Clean up
        del volume, prob, binary
        gc.collect()
        torch.cuda.empty_cache()
    
    # Create zip
    zip_path = output_dir / "submission.zip"
    print(f"\nCreating {zip_path}...")
    
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for vol_file in test_files:
            vid = vol_file.stem
            pred_file = output_dir / f"{vid}.tif"
            if pred_file.exists():
                zf.write(pred_file, f"{vid}.tif")
                print(f"  Added {vid}.tif")
    
    print(f"\nSubmission saved to {zip_path}")
    return zip_path

print("Submission function ready")

In [ ]:
# =============================================================================
# CELL 9: RUN SUBMISSION
# =============================================================================

# Generate submission with optimized params
if cfg.TEST_IMAGES_DIR.exists():
    zip_path = generate_submission(
        model,
        cfg.TEST_IMAGES_DIR,
        cfg.OUTPUT_DIR,
        postprocess_params={
            'threshold': 0.30,
            'use_hysteresis': True,
            'hysteresis_low': 0.16,
            'hysteresis_high': 0.38,
            'min_component_size': 50,
            'closing_iters': 0,
        }
    )
    print(f"\nDone! Submit: {zip_path}")
else:
    print(f"Test directory not found: {cfg.TEST_IMAGES_DIR}")
    print("Update cfg.TEST_IMAGES_DIR to run submission")

In [ ]:
# =============================================================================
# CELL 10: LB IMPROVEMENT TIPS
# =============================================================================

print("""
╔══════════════════════════════════════════════════════════════════════╗
║                    LB IMPROVEMENT TIPS                                ║
╠══════════════════════════════════════════════════════════════════════╣
║ Current: 0.550 → Target: 0.58+                                        ║
╠══════════════════════════════════════════════════════════════════════╣
║ 1. THRESHOLD TUNING                                                   ║
║    - Default 0.5 is usually too high                                  ║
║    - Try: 0.25, 0.30, 0.35 to preserve thin structures                ║
║    - Use validation sweep to find optimal                             ║
╠══════════════════════════════════════════════════════════════════════╣
║ 2. HYSTERESIS THRESHOLDING                                            ║
║    - Preserves connected thin structures                              ║
║    - Good for TopoScore (30% of LB)                                   ║
║    - Recommended: low=0.16, high=0.38                                 ║
╠══════════════════════════════════════════════════════════════════════╣
║ 3. OVERLAP                                                            ║
║    - 0.5 overlap reduces tile artifacts                               ║
║    - Better VOI score (35% of LB)                                     ║
║    - Slower but worth it                                              ║
╠══════════════════════════════════════════════════════════════════════╣
║ 4. SMALL COMPONENT REMOVAL                                            ║
║    - Remove noise < 50-100 voxels                                     ║
║    - Better SurfaceDice (35% of LB)                                   ║
╠══════════════════════════════════════════════════════════════════════╣
║ 5. TTA (if time allows)                                               ║
║    - Z-flip averaging can help slightly                               ║
║    - Set USE_TTA = True                                               ║
╠══════════════════════════════════════════════════════════════════════╣
║ 6. AVOID                                                              ║
║    - closing_iters > 1 (can bridge separate structures)               ║
║    - threshold > 0.5 (loses thin connections)                         ║
║    - min_component_size > 200 (removes valid small structures)        ║
╚══════════════════════════════════════════════════════════════════════╝

Expected LB improvement from postprocessing: +0.01 to +0.03
""")